# Agentic CFI Showcase
Cells showcase four example programs (global FP, struct of functions, dispatch table, FP as function parameter), and then a full pipeline test on the `tinyexpr` (recursive-descent parser) source code, which has plenty of indirect function calls.

## Checking Correctness
`Blue agent successfully protected code.`
`Blue agent allows correct execution of functions.`

The above two lines mean that the agent successfully generated a narrow enough target set to avoid the backdoor, but a large enough target set for correct execution. If both lines are displayed, that means the agent executed correctly! The former line is tested by running a file in each directory called `eval_tests.sh` which the agent cannot see, and simulates a backdoor exploit. The latter is a dynamic execution tool given to the agent to allow it to check execution.

## Agentic Interaction
The agent can additionally be prompted with feedback in the pipeline, based on the number of steps and ultimate user goals. It can be passed as a string that gets added to the agent's prompt, which helps it build further. I primarily used it when testing multi-agent feedback, and this can also be adapted to enable interaction and back-and-forth with agents.

### Example 1: Global FP
Example 1 is a toy example that utilizes a singular global function pointer, which then is invoked. The function pointer may point to any of three semantically permitted functions which are dynamically executed in `tests.sh`, and may not point to any other functions.

The agent correctly deduces the minimal target set and creates the correct policy.cfi.json file, which creates the hardened binary immune to forward-edge hijacking!

In [ ]:
!python3 pipeline.py targets/example1
!git checkout blue/example1
!cat targets/example1/policy.cfi.json

Already on 'main'
Your branch is up to date with 'origin/main'.
Switched to and reset branch 'blue/example1'
[Step 0], agent calls list_c_files()
[Step 1], agent calls read_file()
[Step 2], agent calls compile_to_llvm()
[Step 3], agent calls find_indirect_calls()
[Step 4], agent calls find_pointer_assignments()
[Step 5], agent calls find_function_declarations()
[Step 6], agent calls find_address_taken_functions()
[Step 7], agent calls write_policy()
[Step 8], agent calls compile_llvm_pass()
[Step 8], agent calls policy_to_llvm_pass()
[Step 9], agent calls run_tests()
[Step 10], agent calls write_policy()
[Step 11], agent calls policy_to_llvm_pass()
[Step 12], agent calls run_tests()
[Step 13], agent calls write_log()
Rate limited, sleeping 30s
[Agent done]
[blue/example1 003ebb0] blue agent: example1
 3 files changed, 169 insertions(+)
 create mode 100644 .DS_Store
 create mode 100644 showcase.ipynb
 create mode 100644 targets/example1/policy.cfi.json
Blue agent successfully protected 

In [ ]:
!git checkout main
!cat logs/example1.log

M	showcase.ipynb
Already on 'main'
Your branch is up to date with 'origin/main'.
CFI policy successfully enforced for 'example1'.

Test Results:
- Safe route: SAFE handler reached.
- Admin route: ADMIN handler reached.
- Attack route: Blocked as expected.

All legitimate tests passed, attack prevented through tighter target set.


### Example 2: Struct of Functions (Factory Design Pattern)
Example 2 is a toy example that utilizes the Factory Design Pattern in the form of a struct of function pointers, whose FP fields may point to a set of various functions. The agent must correctly deduce (based on semantics and dynamic execution) what the struct of FPs is allowed to point to, then patch it.

The agent once again correctly deduces the correct target set!

In [ ]:
!python3 pipeline.py targets/example2
!git checkout blue/example2
!cat targets/example2/policy.cfi.json

[main abdf341] added to notebook
 1 file changed, 2 insertions(+), 1 deletion(-)
Already on 'main'
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)
Switched to and reset branch 'blue/example2'
[Step 0], agent calls list_c_files()
[Step 1], agent calls compile_to_llvm()
[Step 2], agent calls find_indirect_calls()
[Step 2], agent calls find_function_pointer_declarations()
[Step 2], agent calls find_pointer_assignments()
[Step 2], agent calls find_function_declarations()
[Step 3], agent calls find_address_taken_functions()
[Step 4], agent calls write_policy()
[Step 5], agent calls compile_llvm_pass()
[Step 6], agent calls policy_to_llvm_pass()
[Step 7], agent calls run_tests()
[Step 8], agent calls write_log()
[Agent done]
[blue/example2 52876c5] blue agent: example2
 2 files changed, 62 insertions(+), 4 deletions(-)
 create mode 100644 targets/example2/policy.cfi.json
Blue agent successfully protected code.
Blue agent allows correct exec

In [ ]:
!cat logs/example2.log

CFI Policy was successfully implemented for the target 'example2'.

### Indirect Call Sites and Policies:
- **Indirect Call at Index 0 in `main`:**
  - Allowed Targets: `admin_connect`, `http_connect`

- **Indirect Call at Index 1 in `main`:**
  - Allowed Targets: `admin_close`, `http_close`

### Testing Results:
- **Output:**
  Testing http route.
  HTTP: connected
  HTTP: closed
  Testing admin route.
  ADMIN: connected
  ADMIN: closed

  Normal tests passed!

### Conclusion:
The instrumented binary successfully passed all functional tests. This suggests that the CFI policy was correctly applied and enforced expected security postures against indirect calls defined in the `example2` project.
No failed tests were encountered, and the hardened binary's functionality aligns with expected behaviors.


### Example 3: Dispatch Table + Multi-File Testing
Example 3 is a toy example that contains multiple .c source files, one defining main and the dispatch function, another defining handlers, and another defining the dispatch table, therefore stressing the agent's overall reasoning. Additionally, the indirect function calls occur through a dispatch table, which connect strings to function invocations, which the agent can rather easily semantically reason over.

The agent correctly deduces the target set once more here.

In [5]:
!python3 pipeline.py targets/example3
!git checkout blue/example3
!cat targets/example3/policy.cfi.json

M	showcase.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)
M	showcase.ipynb
Switched to and reset branch 'blue/example3'
[Step 0], agent calls list_c_files()
[Step 1], agent calls compile_to_llvm()
[Step 1], agent calls compile_to_llvm()
[Step 1], agent calls compile_to_llvm()
[Step 2], agent calls find_indirect_calls()
[Step 2], agent calls find_indirect_calls()
[Step 2], agent calls find_indirect_calls()
[Step 3], agent calls find_pointer_assignments()
[Step 3], agent calls find_pointer_assignments()
[Step 3], agent calls find_pointer_assignments()
[Step 4], agent calls find_function_declarations()
[Step 4], agent calls find_function_declarations()
[Step 4], agent calls find_function_declarations()
[Step 5], agent calls find_address_taken_functions()
[Step 6], agent calls write_policy()
[Step 7], agent calls compile_llvm_pass()
[Step 8], agent calls policy_to_llvm_pass()
[Step 9], agent calls run_tests()
[Ste

In [ ]:
!cat logs/example3.log

CFI Analysis Report for 'targets/example3':

Indirect Call Sites Analysis:
- Indirect call found in function 'dispatch' at index 0.

Function Address-taken Analysis:
- Address-taken functions: ['debug_dump', 'dispatch', 'handle_get', 'handle_post', 'handle_put', 'main', 'puts', 'strcmp']

Constructed Policy:
- For function 'dispatch' call index 0, allowed targets: ['handle_get', 'handle_post', 'handle_put']

Test Results:
- The hardened binary successfully passed all provided tests without any issues.
- stdout: Testing GET route.\nhandled the GET request\nTesting POST route.\nhandled the POST request\nTesting PUT route.\nhandled the PUT request\n\nNormal tests passed!\n
- stderr: (none)

Conclusion:
- All tests passed successfully, and the CFI policy ensures that only allowed handlers can be indirectly called through the 'dispatch' function.


### Example 4: FP as Function Parameter
Example 4 is the last toy example, which passes a function pointer as the parameter of a function. This last test attempts to see the reasoning of the model, because it must reconcile LLVM's SSA IR and Clang's AST, to properly identify what targets are legal statically, as well as dynamically.

The agent deduces the target set correctly once more.

In [7]:
!python3 pipeline.py targets/example4
!git checkout blue/example4
!cat targets/example4/policy.cfi.json

M	showcase.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)
M	showcase.ipynb
Switched to and reset branch 'blue/example4'
[Step 0], agent calls list_c_files()
[Step 1], agent calls compile_to_llvm()
[Step 1], agent calls find_function_declarations()
[Step 1], agent calls find_function_pointer_typedefs()
[Step 1], agent calls find_function_pointer_declarations()
[Step 1], agent calls find_pointer_assignments()
[Step 2], agent calls find_indirect_calls()
[Step 3], agent calls find_address_taken_functions()
[Step 4], agent calls write_policy()
[Step 5], agent calls compile_llvm_pass()
[Step 6], agent calls policy_to_llvm_pass()
[Step 7], agent calls run_tests()
[Step 8], agent calls write_log()
[Agent done]
[blue/example4 217c44b] blue agent: example4
 2 files changed, 212 insertions(+), 12 deletions(-)
 create mode 100644 targets/example4/policy.cfi.json
Blue agent successfully protected code.
Blue agent allows co

In [ ]:
!cat logs/example4.log

Forward-Edge CFI Analysis Report for 'targets/example4'

### Indirect Call Analysis:
- **Function:** with_logging
- **Call Index:** 0
- **Allowed Targets:** add, mul, sub

### Testing Results:
- **Passed Tests:** All functional operations returned expected results without errors or crashes.
- **Standard Output:**
  ```
  Testing add
  op(10, 10) = 20
  Testing sub
  op(10, 10) = 0
  Testing mul
  op(10, 10) = 100

  Normal tests passed!
  ```
- **Standard Error:** No errors reported.

### Conclusion:
The CFI policy is effective, correctly enforcing valid targets at indirect call sites with no broken functionality. All potential vulnerabilities via indirect calls are secured.

### TinyExpr (Real-World Test Case)

The real-world example, TinyExpr is a recursive-descent parser for mathematical functions. With ~600 lines of source code, it utilizes a function called `te_eval` to evaluate different terms, and to do so, utilizes multiple indirect function invocations for convenience. 

There are 16 indirect function call sites in te_eval, which stresses agent reasoning and building exact test sites. Additionally, tinyexpr links multiple external libraries, e.g. `<math.h>`, which forces the agent to reason through tools about what external functions are permissible. Overall, there should be about 5-15 permitted functions per call site, with some varying, as each call site is arity-based.

The Agent has mixed success on these -- there are cases where the agent entirely fails, picking maybe 15 incorrect functions per indirect call site, and it picks some functions that are entirely irrelevant to execution (e.g., places other tinyexpr functions in the target set, which should not be there).

However, in the case shown in the notebook, the agent surprisingly reasons well, but still insufficiently. It generates a full target set for each target site, but with the main caveat being that it is the same target set for each site, so it reasons only once. It then correctly generates a target set, but this target set is too large -- the agent hallucinates function calls that should not be there, e.g., __memcpy_chk, which should not be part of any target set whatsoever.

Thus, the agent has varying levels of success with TinyExpr. Sometimes, the agent successfully acquires a target set (although it's absolutely not perfect), and other times, it entirely fails. The example shown in this notebook is a success, blocking the intended backdoor, but it is not entirely perfect of a target set.

#### Agent Resilience
The agent additionally demonstrates adaptation and resilience (as detailed in the spec, the agent must iterate). When constructing the target set, it runs the dynamic tests three times, and reconsiders its policies thrice before being satisfied. 

Despite not building the perfect set, it still constructs a correct target set here. Therefore, the agent correctly showcases adaptation and resilience.

In [ ]:
!cat logs/tinyexpr.log

CFI Analysis for tinyexpr Project Completed Successfully

1. **Analysis Overview**: 
   - Source files involved: `tinyexpr.c` and `driver.c`.
   - Indirect call sites identified in `te_eval` function.
   - Utilized function pointers engaged in evaluation contexts.

2. **CFI Policy Formulation**: 
   - Policy created targeting indirect call resolution for plausible function sets.
   - Employed a comprehensive targeting of functions recognized via address-taking across project IR.

3. **Testing Summary**: 
   - Performed iterative tests via `tests.sh` in `targets/tinyexpr`.
   - Achieved test pass with outputs:
     - Arithmetic and built-in evaluations recorded expected results (14 and 4 respectively).
     - Custom functions validated delivering outputs 1 and 49.
     - "Normal tests passed!" confirms functionality preservation and adequate CFI coverage.

4. **Security Assurance**: 
   - Tightened function checks ensured no indirect invocation outside defined bounds.
   - Functional as

In [9]:
!python3 pipeline.py targets/tinyexpr
!git checkout blue/tinyexpr
!cat targets/tinyexpr/policy.cfi.json

M	showcase.ipynb
Already on 'main'
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)
M	showcase.ipynb
Switched to and reset branch 'blue/tinyexpr'
[Step 0], agent calls list_c_files()
[Step 1], agent calls compile_to_llvm()
[Step 1], agent calls compile_to_llvm()
[Step 1], agent calls find_function_declarations()
[Step 1], agent calls find_function_declarations()
[Step 2], agent calls find_indirect_calls()
[Step 2], agent calls find_indirect_calls()
[Step 2], agent calls find_function_pointer_typedefs()
[Step 2], agent calls find_function_pointer_typedefs()
[Step 2], agent calls find_function_pointer_declarations()
[Step 2], agent calls find_function_pointer_declarations()
[Step 2], agent calls find_pointer_assignments()
[Step 2], agent calls find_pointer_assignments()
[Step 3], agent calls find_address_taken_functions()
[Step 4], agent calls write_policy()
[Step 5], agent calls compile_llvm_pass()
[Step 6], agent calls policy_to_llvm_p